# 28. RAG 파이프라인 구축

> **제28장** · **이론편 대응: 21장 (RAG)**
> **예상 소요**: 80분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: **chromadb** (1절 참조)
> **API 키**: 선택 (없으면 로컬 모델로 대체)

---

## 이 장에서 하는 일

27장에서 만든 검색기에 **LLM을 붙인다.** 이론편 25장의 전체 구조를 완성하는 것이다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | **준비 — chromadb 설치** | — |
| 2 | RAG가 필요한 이유 — 직접 확인 | 21.1절 |
| 3 | 벡터 DB 사용 | 21.3절 |
| 4 | **전체 파이프라인 조립** | 21.2절 |
| 5 | **검색 실패가 답변에 미치는 영향** ★ | 21.4절 |
| 6 | 프롬프트 설계 | 21.4절 |
| 7 | 출처 표시 | 21.4절 |
| 8 | 한계와 개선 방향 | 21.5절 |

**5절이 이 장의 핵심이다.** RAG는 "검색 + 생성"인데,
검색이 틀리면 생성이 아무리 좋아도 소용없다는 것을 직접 확인한다.

---

## 1. 준비 — chromadb 설치

### 왜 벡터 DB가 필요한가

27장 6절에서 `SimpleVectorStore`를 만들었다. 문서 8개로는 충분했지만
실제로는 부족하다.

| 항목 | 직접 구현 | 벡터 DB |
|---|---|---|
| 검색 방식 | 전부 비교 (선형 탐색) | 근사 최근접 탐색 |
| 문서 10만 개 | 느림 | 빠름 |
| 저장 | 메모리에만 | 디스크에 영구 저장 |
| 메타데이터 | 직접 관리 | 함께 저장·필터링 |
| 추가·삭제 | 전체 재구축 | 개별 처리 |

### 이 장에서 쓸 것

**Chroma** — 설치가 간단하고 로컬에서 바로 쓸 수 있다.

```
pip install chromadb
```

다른 선택지도 있다.

| 이름 | 특징 |
|---|---|
| Chroma | 간단, 로컬 실행 (이 장) |
| FAISS | 빠름, Meta 제작, 저장은 직접 관리 |
| Qdrant, Weaviate | 서버형, 대규모용 |
| pgvector | PostgreSQL 확장 |

**어느 것을 쓰든 원리는 27장에서 만든 것과 같다.**

In [ ]:
import importlib

print("=" * 60)
print("필요 패키지 확인")
print("=" * 60)

required = [
    ("chromadb", "벡터 데이터베이스", "pip install chromadb"),
    ("sentence_transformers", "임베딩 (27장)", "pip install sentence-transformers"),
    ("numpy", "수치 계산", "pip install numpy"),
]

missing = []
for name, desc, install in required:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "설치됨")
        print(f"[OK]   {name:<24}{ver:<12}{desc}")
    except ImportError:
        print(f"[없음] {name:<24}{'':<12}{desc}")
        missing.append(install)

# API는 선택 사항
print()
try:
    import openai
    print(f"[선택] openai (API 사용 시)  {openai.__version__}")
except ImportError:
    print("[선택] openai 미설치 — 로컬 모델로 대체 가능")

print("-" * 60)
if missing:
    print("설치가 필요합니다:")
    for cmd in set(missing):
        print(f"  {cmd}")
else:
    print("[준비 완료] 2절로 진행하세요.")

In [ ]:
import numpy as np
import os
from pathlib import Path

# 27장에서 쓴 임베딩 모델
from sentence_transformers import SentenceTransformer

print("임베딩 모델 불러오는 중... (27장에서 받았다면 즉시 로드)")
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print(f"완료 — 차원 {embedder.get_sentence_embedding_dimension()}")

# API 키 확인 (25장과 같은 방식)
root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent
try:
    from dotenv import load_dotenv
    load_dotenv(root / ".env")
except ImportError:
    pass

API_KEY = None
BASE_URL = None
for env_name, base in [("OPENAI_API_KEY", None),
                       ("GROQ_API_KEY", "https://api.groq.com/openai/v1"),
                       ("GEMINI_API_KEY", "https://generativelanguage.googleapis.com/v1beta/openai/")]:
    if os.getenv(env_name):
        API_KEY, BASE_URL = os.getenv(env_name), base
        print(f"API 키 발견: {env_name}")
        break

if not API_KEY:
    print("API 키 없음 — 생성 부분은 로컬 모델 또는 안내로 대체합니다.")

---

## 2. RAG가 필요한 이유 — 이론편 21.1절

이론편 21.1절에서 세 가지 문제를 들었다.

1. **최신 정보를 모른다** — 학습 시점 이후를 알 수 없다
2. **내부 문서를 모른다** — 회사 규정 같은 것은 학습에 없다
3. **없는 것을 지어낸다** — 모른다고 하지 않고 그럴듯하게 답한다

**세 번째를 직접 확인해 보자.** 존재하지 않는 것을 물어보면 어떻게 될까.

In [ ]:
def ask_llm(messages, max_tokens=200, temperature=0.3):
    """LLM 호출 — API 키가 없으면 None"""
    if not API_KEY:
        return None
    try:
        from openai import OpenAI
        kwargs = {"api_key": API_KEY}
        if BASE_URL:
            kwargs["base_url"] = BASE_URL
        client = OpenAI(**kwargs)

        models = {"https://api.groq.com/openai/v1": "llama-3.3-70b-versatile",
                  "https://generativelanguage.googleapis.com/v1beta/openai/": "gemini-2.0-flash"}
        model = models.get(BASE_URL, "gpt-4o-mini")

        r = client.chat.completions.create(
            model=model, messages=messages,
            max_tokens=max_tokens, temperature=temperature)
        return r.choices[0].message.content
    except Exception as e:
        print(f"[오류] {type(e).__name__}: {str(e)[:120]}")
        return None


print("=" * 65)
print("존재하지 않는 것을 물어보면 (이론편 21.1절)")
print("=" * 65)

fake_question = "우리 회사의 2026년 재택근무 규정을 알려주세요."

print(f"질문: {fake_question}")
print()

answer = ask_llm([{"role": "user", "content": fake_question}])

if answer:
    print("[RAG 없이] 모델의 답변")
    print(f"  {answer}")
    print()
    print("-" * 65)
    print("모델은 '우리 회사'가 어디인지 모른다.")
    print("그런데도 답을 만들어 낸다 — 이것이 환각(hallucination)이다.")
else:
    print("[건너뜀] API 키가 없어 실제 호출을 하지 않습니다.")
    print()
    print("이 실험의 요지")
    print("  모델은 '우리 회사'가 어디인지 알 수 없다.")
    print("  그런데도 '모른다'고 하지 않고 그럴듯한 규정을 지어내는 경우가 많다.")
    print("  → 이론편 21.1절에서 다룬 환각 문제")

### RAG의 해법

**모델에게 답을 지어내게 하지 말고, 근거를 함께 준다.**

```
[기존]  질문 → LLM → 답변

[RAG]   질문 → 검색 → 관련 문서 ─┐
                                 ├→ LLM → 답변
                          질문 ──┘
```

핵심은 프롬프트를 이렇게 바꾸는 것이다.

```
다음 문서를 참고해 질문에 답하세요.
문서에 없는 내용은 "문서에서 찾을 수 없습니다"라고 답하세요.

[문서]
...검색된 내용...

[질문]
...사용자 질문...
```

**"없으면 없다고 하라"는 지시**가 환각을 크게 줄인다. 6절에서 이 프롬프트를 다듬는다.

---

## 3. 벡터 DB 사용 — 이론편 21.3절

Chroma로 27장의 `SimpleVectorStore`를 대체한다.

In [ ]:
import chromadb
import numpy as np

# 실습용 사내 문서 (모델이 학습했을 리 없는 내용)
company_docs = [
    "재택근무는 주 2회까지 신청할 수 있으며, 팀장 승인이 필요하다. 신청은 최소 3일 전에 해야 한다.",
    "연차는 입사 1년 미만은 월 1일씩 발생하고, 1년 이상은 연 15일이 부여된다.",
    "출장비는 국내 일 8만원, 해외 일 15만원까지 정산 가능하다. 영수증 첨부가 필수다.",
    "교육비는 연간 200만원까지 지원되며, 업무 관련성이 인정되어야 한다.",
    "점심 시간은 12시부터 1시까지이며, 팀 사정에 따라 30분 조정할 수 있다.",
    "장비 신청은 사내 포털에서 하며, 노트북은 3년마다 교체 대상이 된다.",
    "경조사비는 결혼 50만원, 조사 30만원이 지급된다. 신청서를 제출해야 한다.",
    "야근 식대는 저녁 8시 이후 근무 시 1만원까지 지원된다.",
]

print("=" * 65)
print("벡터 DB 구축")
print("=" * 65)

# 메모리에서 실행 (파일로 저장하려면 PersistentClient 사용)
client = chromadb.Client()

# 같은 이름이 있으면 지우고 새로 만든다
try:
    client.delete_collection("company_policy")
except Exception:
    pass

collection = client.create_collection(
    name="company_policy",
    metadata={"description": "사내 규정 문서"},
)

# 임베딩을 직접 계산해 넣는다 (27장과 같은 방식)
embeddings = embedder.encode(company_docs, normalize_embeddings=True)

collection.add(
    ids=[f"doc_{i}" for i in range(len(company_docs))],
    embeddings=embeddings.tolist(),
    documents=company_docs,
    metadatas=[{"category": "규정", "doc_id": i} for i in range(len(company_docs))],
)

print(f"저장된 문서: {collection.count()}개")
print(f"임베딩 차원: {embeddings.shape[1]}")
print()
print("메타데이터도 함께 저장된다 — 나중에 필터링에 쓸 수 있다.")

In [ ]:
import numpy as np

print("=" * 70)
print("검색 — Chroma는 '거리'를 돌려준다")
print("=" * 70)

query = "재택근무 신청 방법"
q_emb = embedder.encode([query], normalize_embeddings=True)

results = collection.query(
    query_embeddings=q_emb.tolist(),
    n_results=3,
)

print(f"질문: {query}")
print()
print(f"{'순위':<6}{'거리':<12}{'코사인(환산)':<16}{'문서'}")
print("-" * 70)
for rank, (doc, dist) in enumerate(zip(results["documents"][0],
                                        results["distances"][0]), 1):
    # 정규화된 벡터에서 squared L2 거리 = 2(1 - cos)
    cos = 1 - dist / 2
    print(f"{rank:<6}{dist:<12.4f}{cos:<16.4f}{doc[:34]}...")
print("-" * 70)
print()

# 직접 계산한 코사인과 대조
sims = embeddings @ q_emb[0]
top_direct = np.sort(sims)[::-1][:3]
print("27번 방식으로 직접 계산한 코사인")
print(f"  {top_direct.round(4)}")
print()
converted = [1 - d/2 for d in results["distances"][0]]
print("Chroma 거리를 코사인으로 환산")
print(f"  {np.array(converted).round(4)}")
print()
assert np.allclose(sorted(converted, reverse=True), top_direct, atol=1e-3)
print("[OK] 같은 값 — 표현 방식만 다르다")
print()
print("Chroma 기본 거리는 squared L2 다.")
print("  정규화된 벡터에서는  거리 = 2(1 - 코사인)  관계가 성립한다.")
print("  따라서  코사인 = 1 - 거리/2")

---

## 4. 전체 파이프라인 조립 — 이론편 21.2절

이제 검색과 생성을 연결한다. **RAG의 전체 흐름**은 이렇다.

1. 질문을 벡터로 변환
2. 벡터 DB에서 관련 문서 검색
3. 검색 결과를 프롬프트에 넣기
4. LLM이 답변 생성

In [ ]:
import numpy as np


class SimpleRAG:
    # RAG 파이프라인 (이론편 21.2절)

    def __init__(self, collection, embedder, top_k=3):
        self.collection = collection
        self.embedder = embedder
        self.top_k = top_k

    def retrieve(self, question, top_k=None):
        """1~2단계: 질문을 벡터로 바꿔 관련 문서를 찾는다"""
        k = top_k or self.top_k
        q_emb = self.embedder.encode([question], normalize_embeddings=True)
        res = self.collection.query(query_embeddings=q_emb.tolist(), n_results=k)

        return [
            {"text": doc, "score": 1 - dist / 2, "id": doc_id}
            for doc, dist, doc_id in zip(
                res["documents"][0], res["distances"][0], res["ids"][0])
        ]

    def build_prompt(self, question, docs):
        """3단계: 검색 결과를 프롬프트에 넣는다"""
        context = "\n\n".join(
            f"[문서 {i+1}] {d['text']}" for i, d in enumerate(docs))

        system = (
            "당신은 사내 규정을 안내하는 도우미입니다.\n"
            "아래 제공된 문서만을 근거로 답하세요.\n"
            "문서에 없는 내용은 '제공된 문서에서 찾을 수 없습니다'라고 답하세요.\n"
            "추측하거나 일반 상식으로 보충하지 마세요."
        )
        user = f"[참고 문서]\n{context}\n\n[질문]\n{question}"

        return [{"role": "system", "content": system},
                {"role": "user", "content": user}]

    def answer(self, question, verbose=True):
        """전체 실행"""
        docs = self.retrieve(question)

        if verbose:
            print(f"[검색된 문서 {len(docs)}개]")
            for i, d in enumerate(docs, 1):
                print(f"  {i}. [{d['score']:.4f}] {d['text'][:44]}...")
            print()

        messages = self.build_prompt(question, docs)
        answer = ask_llm(messages, max_tokens=300)

        return {"question": question, "docs": docs,
                "messages": messages, "answer": answer}


rag = SimpleRAG(collection, embedder, top_k=3)

print("=" * 70)
print("RAG 파이프라인 실행")
print("=" * 70)

result = rag.answer("재택근무는 며칠까지 가능한가요?")

if result["answer"]:
    print("[답변]")
    print(f"  {result['answer']}")
else:
    print("[건너뜀] API 키가 없어 생성은 하지 않았습니다.")
    print("검색까지는 정상 동작했습니다 — 위의 검색 결과를 확인하세요.")

In [ ]:
print("=" * 70)
print("실제로 LLM에게 전달되는 프롬프트")
print("=" * 70)

for msg in result["messages"]:
    print(f"\n[{msg['role']}]")
    print(msg["content"])

print()
print("=" * 70)
print("구조 확인")
print(f"  system : 규칙 지시 ({len(result['messages'][0]['content'])}자)")
print(f"  user   : 검색 문서 + 질문 ({len(result['messages'][1]['content'])}자)")
print()
print("24~25장에서 다룬 ChatML 형식 그대로다.")
print("검색 결과가 user 메시지에 들어간다는 점만 다르다.")

In [ ]:
print("=" * 70)
print("여러 질문으로 확인")
print("=" * 70)

questions = [
    "연차는 몇 일인가요?",
    "해외 출장비 한도는?",
    "노트북은 언제 바꿔주나요?",
    "주차비도 지원되나요?",        # 문서에 없는 내용
]

for q in questions:
    print(f"\n{'='*70}")
    print(f"질문: {q}")
    print("-" * 70)
    r = rag.answer(q, verbose=False)

    print(f"검색 1위: [{r['docs'][0]['score']:.4f}] {r['docs'][0]['text'][:44]}...")
    if r["answer"]:
        print(f"답변    : {r['answer']}")
    else:
        print("답변    : (API 키 없음)")

print()
print("=" * 70)
print("마지막 질문('주차비')에 주목")
print("  문서에 없는 내용이므로 '찾을 수 없다'고 답해야 정상이다.")
print("  system 프롬프트의 지시가 작동하는지 확인하는 지점이다.")

---

## 5. 검색 실패가 답변에 미치는 영향 ★ — 이론편 21.4절

**RAG의 가장 중요한 성질**을 확인할 차례다.

> RAG의 답변 품질은 **검색 품질을 넘을 수 없다.**

검색이 엉뚱한 문서를 가져오면, LLM은 그것을 근거로 답한다.
LLM이 아무리 좋아도 소용없다.

In [ ]:
import numpy as np

print("=" * 70)
print("검색이 실패하는 경우")
print("=" * 70)

# 구어체·짧은 질문은 검색이 어렵다
tricky_queries = [
    ("재택근무 신청 방법", 0),        # 정답: 0번 문서
    ("집에서 일하려면?", 0),          # 같은 뜻, 다른 표현
    ("휴가", 1),                      # 너무 짧음
    ("연차 며칠?", 1),
]

print(f"{'질문':<22}{'기대':<8}{'실제 1위':<10}{'점수':<10}{'판정'}")
print("-" * 70)

for query, expected_idx in tricky_queries:
    docs = rag.retrieve(query, top_k=1)
    found_id = docs[0]["id"]
    found_idx = int(found_id.split("_")[1])
    ok = "정확" if found_idx == expected_idx else "틀림"
    print(f"{query:<22}{expected_idx:<8}{found_idx:<10}{docs[0]['score']:<10.4f}{ok}")

print("-" * 70)
print()
print("짧거나 구어체인 질문에서 검색이 어긋날 수 있다.")
print("27장 5절에서 본 것처럼, 임베딩은 표현 방식에 영향을 받는다.")

In [ ]:
print("=" * 70)
print("일부러 틀린 문서를 주면 어떻게 되나")
print("=" * 70)

question = "연차는 며칠인가요?"

# 올바른 문서
correct_docs = [{"text": company_docs[1], "score": 0.9, "id": "doc_1"}]
# 관련 없는 문서
wrong_docs = [{"text": company_docs[6], "score": 0.9, "id": "doc_6"}]

for label, docs in [("올바른 문서", correct_docs), ("엉뚱한 문서", wrong_docs)]:
    print(f"\n[{label}]")
    print(f"  제공: {docs[0]['text'][:50]}...")
    msgs = rag.build_prompt(question, docs)
    ans = ask_llm(msgs, max_tokens=200)
    if ans:
        print(f"  답변: {ans}")
    else:
        print("  답변: (API 키 없음)")

print()
print("=" * 70)
print("이것이 뜻하는 것 (이론편 21.4절)")
print()
print("  엉뚱한 문서를 받으면 LLM은 두 가지 중 하나를 한다:")
print("    1) '문서에서 찾을 수 없다'고 답한다  ← 프롬프트가 잘 작동한 경우")
print("    2) 받은 문서로 어떻게든 답을 만든다  ← 잘못된 답변 생성")
print()
print("  어느 쪽이든 **검색이 틀리면 좋은 답을 얻을 수 없다.**")
print()
print("  → RAG 개선은 대부분 '검색 개선'이다. (27장 8절 참조)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

# top_k 에 따른 검색 성공률
eval_pairs = [
    ("재택근무 신청 방법", 0),
    ("연차 일수", 1),
    ("출장비 한도", 2),
    ("교육비 지원", 3),
    ("점심 시간", 4),
    ("노트북 교체 주기", 5),
    ("경조사비", 6),
    ("야근 식대", 7),
]

print("=" * 65)
print("top_k 에 따른 검색 성공률 (이론편 21.5절)")
print("=" * 65)

ks = [1, 2, 3, 5]
recalls = []
for k in ks:
    hits = 0
    for q, expected in eval_pairs:
        docs = rag.retrieve(q, top_k=k)
        ids = [int(d["id"].split("_")[1]) for d in docs]
        if expected in ids:
            hits += 1
    recalls.append(hits / len(eval_pairs))

print(f"{'top_k':<10}{'Recall':<12}{'설명'}")
print("-" * 65)
for k, r in zip(ks, recalls):
    print(f"{k:<10}{r:<12.4f}상위 {k}개 안에 정답이 있는 비율")
print("-" * 65)
print()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, recalls, marker="o", linewidth=2.5, color="#1E40AF")
ax.set_xlabel("top_k (가져올 문서 수)")
ax.set_ylabel("Recall")
ax.set_title("top_k 와 검색 성공률")
ax.set_ylim(0, 1.05)
ax.set_xticks(ks)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("top_k 를 늘리면 정답을 놓칠 확률은 줄어든다.")
print()
print("하지만 무작정 늘릴 수 없다:")
print("  1) 관련 없는 문서가 섞여 LLM이 혼란스러워진다")
print("  2) 프롬프트가 길어져 비용이 는다 (25장)")
print("  3) 문맥 창 한계에 걸린다 (이론편 20.4절)")
print()
print("보통 3~5개를 쓰고, 재순위화로 정확도를 높인다.")

---

## 6. 프롬프트 설계 — 이론편 21.4절

같은 검색 결과라도 **프롬프트를 어떻게 쓰느냐**에 따라 답변이 달라진다.

세 가지를 비교해 보자.

In [ ]:
print("=" * 70)
print("프롬프트 설계 비교")
print("=" * 70)

question = "주차비도 지원되나요?"      # 문서에 없는 내용
docs = rag.retrieve(question, top_k=2)

print(f"질문: {question}  (문서에 없는 내용)")
print(f"검색된 문서: {docs[0]['text'][:44]}...")
print()

context = "\n\n".join(f"[문서 {i+1}] {d['text']}" for i, d in enumerate(docs))

prompts = {
    "지시 없음": [
        {"role": "user", "content": f"{context}\n\n질문: {question}"},
    ],
    "간단한 지시": [
        {"role": "system", "content": "제공된 문서를 참고해 답하세요."},
        {"role": "user", "content": f"[문서]\n{context}\n\n[질문]\n{question}"},
    ],
    "명확한 지시": [
        {"role": "system", "content":
            "제공된 문서만을 근거로 답하세요.\n"
            "문서에 없는 내용은 '제공된 문서에서 찾을 수 없습니다'라고만 답하세요.\n"
            "추측하거나 일반 상식으로 보충하지 마세요."},
        {"role": "user", "content": f"[문서]\n{context}\n\n[질문]\n{question}"},
    ],
}

for name, msgs in prompts.items():
    print(f"\n[{name}]")
    ans = ask_llm(msgs, max_tokens=150)
    if ans:
        print(f"  {ans}")
    else:
        print("  (API 키 없음)")

print()
print("=" * 70)
print("차이가 나는 지점")
print("  지시가 없으면 모델이 일반 상식으로 답을 채울 수 있다.")
print("  '없으면 없다고 하라'는 지시가 환각을 크게 줄인다.")
print()
print("실무 프롬프트에 넣으면 좋은 것")
print("  - 근거를 문서로 제한")
print("  - 없을 때의 답변 방식 지정")
print("  - 추측 금지 명시")
print("  - 출처 표시 요구 (7절)")

---

## 7. 출처 표시 — 이론편 21.4절

RAG의 장점 중 하나는 **답변의 근거를 보여줄 수 있다**는 것이다.
이론편 21.1절에서 다룬 "검증 가능성"이 여기서 나온다.

사용자가 "이 답이 어디서 나왔지?"를 확인할 수 있으면 신뢰도가 크게 오른다.

In [ ]:
class RAGWithCitation(SimpleRAG):
    # 출처를 함께 표시하는 RAG (이론편 21.4절)

    def build_prompt(self, question, docs):
        context = "\n\n".join(
            f"[{i+1}] {d['text']}" for i, d in enumerate(docs))

        system = (
            "당신은 사내 규정을 안내하는 도우미입니다.\n"
            "아래 문서만을 근거로 답하세요.\n"
            "답변에 사용한 문서 번호를 [1], [2] 형식으로 표시하세요.\n"
            "문서에 없는 내용은 '제공된 문서에서 찾을 수 없습니다'라고 답하세요."
        )
        user = f"[참고 문서]\n{context}\n\n[질문]\n{question}"
        return [{"role": "system", "content": system},
                {"role": "user", "content": user}]

    def answer(self, question, verbose=True):
        result = super().answer(question, verbose=False)

        if verbose:
            print(f"질문: {question}")
            print()
            if result["answer"]:
                print("[답변]")
                print(f"  {result['answer']}")
                print()
            print("[근거 문서]")
            for i, d in enumerate(result["docs"], 1):
                print(f"  [{i}] (유사도 {d['score']:.3f}) {d['text']}")
        return result


rag_cited = RAGWithCitation(collection, embedder, top_k=3)

print("=" * 70)
print("출처를 함께 보여주는 RAG")
print("=" * 70)
r = rag_cited.answer("출장 갈 때 하루에 얼마까지 쓸 수 있나요?")

if not r["answer"]:
    print()
    print("(API 키가 없어 답변은 생성되지 않았습니다)")
    print("검색된 근거 문서는 위에 표시되어 있습니다.")

In [ ]:
print("=" * 70)
print("출처 표시가 중요한 이유")
print("=" * 70)
print()
print("1) 사용자가 검증할 수 있다")
print("   답변이 맞는지 원문을 직접 확인 가능")
print()
print("2) 환각을 발견하기 쉽다")
print("   근거 문서에 없는 내용이 답변에 있으면 바로 보인다")
print()
print("3) 검색 문제를 진단할 수 있다")
print("   답이 이상하면 검색된 문서를 보고 원인을 찾는다")
print()
print("-" * 70)
print("실무에서 자주 쓰는 형태")
print()
print("""
{
  "answer": "국내 출장은 하루 8만원까지 정산 가능합니다 [1].",
  "sources": [
    {"id": "doc_2", "text": "출장비는 국내 일 8만원...", "score": 0.87}
  ]
}
""")
print("24장에서 다룬 구조화된 출력을 쓰면 이런 형태로 받을 수 있다.")

---

## 8. 한계와 개선 방향 — 이론편 21.5절

RAG를 만들어 봤으니 **무엇이 어려운지** 정리한다.

In [ ]:
print("=" * 70)
print("RAG의 주요 실패 유형 (이론편 21.5절)")
print("=" * 70)
print()

failures = [
    ("검색 실패", "관련 문서를 못 찾음",
     "임베딩 모델 교체 / 하이브리드 검색 / 질의 확장"),
    ("분할 문제", "답이 두 조각에 걸쳐 있음",
     "chunk 크기 조정 / overlap 늘리기 / 부모 문서 참조"),
    ("문맥 초과", "문서가 많아 잘림",
     "top_k 조정 / 재순위화 / 요약 후 전달"),
    ("환각", "문서에 없는 내용 생성",
     "프롬프트 강화 / 출처 표시 / 후처리 검증"),
    ("최신성", "저장된 문서가 오래됨",
     "정기적 재색인 / 타임스탬프 관리"),
]

print(f"{'유형':<14}{'증상':<26}{'개선 방향'}")
print("-" * 70)
for name, symptom, fix in failures:
    print(f"{name:<14}{symptom:<26}{fix}")
print("-" * 70)
print()
print("[중요] 대부분의 문제가 **검색 단계**에 있다.")
print("  5절에서 확인했듯 검색이 틀리면 생성이 좋아도 소용없다.")
print("  RAG를 개선할 때는 검색부터 점검하는 것이 순서다.")

In [ ]:
print("=" * 70)
print("개선 기법 개요 (이론편 21.4절)")
print("=" * 70)
print()

print("[1] 하이브리드 검색")
print("  의미 검색 + 키워드 검색을 함께 쓴다.")
print("  27장 4절에서 봤듯 각각 강점이 다르다.")
print("  고유명사·제품코드는 키워드가, 표현 차이는 의미 검색이 유리하다.")
print()

print("[2] 재순위화 (Reranking)")
print("  1단계: 벡터 검색으로 후보 20~50개를 빠르게 추림")
print("  2단계: 정밀한 모델로 다시 순위를 매겨 상위 3~5개 선택")
print("  전체를 정밀 평가하면 느리므로 두 단계로 나눈다.")
print()

print("[3] 질의 재작성 (Query Rewriting)")
print("  '집에서 일하려면?' → '재택근무 신청 절차'")
print("  LLM으로 질문을 검색에 적합한 형태로 바꾼다.")
print("  5절에서 본 구어체 질문 문제를 완화한다.")
print()

print("[4] 부모 문서 검색")
print("  작은 조각으로 검색하되, LLM에게는 그 조각이 속한 큰 문서를 준다.")
print("  검색 정확도와 문맥 충분성을 함께 얻는다.")
print()

print("-" * 70)
print("어느 것을 먼저 시도할까")
print("  1) 먼저 **평가 데이터를 만든다** (27장 8절)")
print("  2) 현재 Recall/MRR 을 측정한다")
print("  3) 하나씩 적용하며 수치가 오르는지 확인한다")
print()
print("측정 없이 기법만 추가하면 나아졌는지 알 수 없다.")

In [ ]:
print("=" * 70)
print("RAG vs 파인튜닝 — 언제 무엇을 (이론편 22.6절 예고)")
print("=" * 70)
print()
print(f"{'상황':<32}{'권장':<12}{'이유'}")
print("-" * 70)
rows = [
    ("최신 정보가 자주 바뀜",       "RAG",     "문서만 갱신하면 됨"),
    ("근거를 보여줘야 함",         "RAG",     "출처 표시 가능"),
    ("문서가 많고 계속 늘어남",     "RAG",     "재학습 불필요"),
    ("특정 말투·형식을 익혀야 함",  "파인튜닝", "지식이 아닌 방식의 문제"),
    ("전문 용어 이해가 필요",       "파인튜닝", "표현 자체를 학습"),
    ("응답 속도가 중요",           "파인튜닝", "검색 단계가 없음"),
]
for a, b, c in rows:
    print(f"{a:<32}{b:<12}{c}")
print("-" * 70)
print()
print("둘은 배타적이지 않다. 함께 쓰는 경우도 많다:")
print("  파인튜닝으로 말투와 형식을 익히고, RAG로 최신 정보를 공급한다.")
print()
print("→ 파인튜닝은 31~32장에서 다룬다.")

---

## 9. 정리

### 만든 것

```
질문 → 임베딩 → 벡터 DB 검색 → 프롬프트 조립 → LLM → 답변 + 출처
```

각 단계에서 확인한 것

| 단계 | 확인 내용 |
|---|---|
| 임베딩 | 27장에서 만든 것 그대로 사용 |
| 벡터 DB | Chroma 거리 = 2(1−코사인) 검증 |
| 검색 | 구어체·짧은 질문에서 실패할 수 있음 |
| 프롬프트 | "없으면 없다고 하라"가 환각을 줄임 |
| 출처 | 검증 가능성 확보 |

### 기억할 것

| 항목 | 요점 |
|---|---|
| **RAG의 상한** | **검색 품질을 넘을 수 없다** |
| Chroma 거리 | squared L2 — 코사인 = 1 − 거리/2 |
| top_k | 3~5개가 보통. 늘리면 혼란·비용 증가 |
| 프롬프트 | 근거 제한 + 없을 때 지시 + 추측 금지 |
| 출처 표시 | 검증·진단·신뢰에 모두 도움 |
| 개선 순서 | **측정 먼저**, 그다음 기법 적용 |

### 이론편 21장을 마치며

이론편에서 배운 RAG의 구조를 전부 구현했다.

| 이론편 절 | 이 장 |
|---|---|
| 21.1 환각 문제 | 2절에서 확인 |
| 21.2 파이프라인 | 4절에서 조립 |
| 21.3 벡터 검색 | 3절 + 22번 |
| 21.4 개선 기법 | 6~8절 |
| 21.5 평가 | 5절 + 27장 8절 |

### 다음 장

**29. Advanced RAG — 검색 품질 끌어올리기** — 이론편 21.4절.
3장에서 확인했듯 RAG의 품질은 검색 품질을 넘을 수 없다.
재순위화·질의 재작성·부모 문서 검색 등 검색 자체를 개선하는 기법들을 직접 구현하고 효과를 측정한다.

### 검색 점수 분포를 그림으로

5절에서 "검색이 틀리면 답변도 틀린다"고 했다.
**그렇다면 검색이 맞았는지 어떻게 판단할까.** 점수 분포를 보면 답이 보인다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- 왼쪽: 관련 질문 vs 무관 질문의 점수 분포 ---
ax = axes[0]

related_queries = [
    "재택근무 신청", "연차 일수", "출장비 한도", "교육비 지원",
    "점심 시간", "노트북 교체", "경조사비", "야근 식대",
]
unrelated_queries = [
    "주차장 이용", "구내식당 메뉴", "헬스장 할인", "셔틀버스 시간",
    "회의실 예약", "명함 신청",
]

related_scores = [rag.retrieve(q, top_k=1)[0]["score"] for q in related_queries]
unrelated_scores = [rag.retrieve(q, top_k=1)[0]["score"] for q in unrelated_queries]

ax.hist(related_scores, bins=8, alpha=0.65,
        label=f"문서에 있는 질문 ({len(related_scores)}개)", color="#0D9488")
ax.hist(unrelated_scores, bins=8, alpha=0.65,
        label=f"문서에 없는 질문 ({len(unrelated_scores)}개)", color="#DC2626")

gap_lo, gap_hi = max(unrelated_scores), min(related_scores)
if gap_hi > gap_lo:
    threshold = (gap_lo + gap_hi) / 2
    ax.axvline(threshold, color="#1E40AF", linestyle="--", linewidth=2)
    ax.text(threshold + 0.005, ax.get_ylim()[1] * 0.75,
            f"임계값 후보\n{threshold:.3f}", fontsize=8, color="#1E40AF")
else:
    threshold = None

ax.set_xlabel("검색 1위 점수")
ax.set_ylabel("질문 수")
ax.set_title("점수만으로 구별할 수 있는가")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- 오른쪽: top_k 에 따른 문맥 길이와 성공률 ---
ax = axes[1]
ks = [1, 2, 3, 5, 8]
context_chars, recalls_k = [], []

for k in ks:
    total_chars = 0
    hits = 0
    for q, gold_idx in eval_pairs:
        docs = rag.retrieve(q, top_k=k)
        total_chars += sum(len(d["text"]) for d in docs)
        ids = [int(d["id"].split("_")[1]) for d in docs]
        if gold_idx in ids:
            hits += 1
    context_chars.append(total_chars / len(eval_pairs))
    recalls_k.append(hits / len(eval_pairs))

ax.plot(ks, recalls_k, marker="o", linewidth=2.5,
        color="#0D9488", label="Recall")
ax.set_xlabel("top_k (LLM 에게 주는 문서 수)")
ax.set_ylabel("Recall", color="#0D9488")
ax.set_ylim(0, 1.1)
ax.set_xticks(ks)
ax.grid(alpha=0.3)

ax2 = ax.twinx()
ax2.plot(ks, context_chars, marker="s", linewidth=2.5,
         color="#EA580C", linestyle="--", label="문맥 길이")
ax2.set_ylabel("프롬프트 평균 길이 (자)", color="#EA580C")

ax.set_title("정확도와 비용의 맞바꿈")
plt.tight_layout()
plt.show()

print("왼쪽: 두 분포가 겹치는 정도가 중요하다")
print(f"  문서에 없는 질문의 최고 점수: {max(unrelated_scores):.4f}")
print(f"  문서에 있는 질문의 최저 점수: {min(related_scores):.4f}")
if threshold:
    print(f"  → 두 분포가 나뉘므로 임계값 {threshold:.3f} 로 거를 수 있다")
else:
    print("  → 겹친다. 점수만으로는 완전히 나눌 수 없다 (07번 정밀도-재현율)")
print()
print("오른쪽: k 를 늘리면 Recall 은 오르지만 프롬프트가 길어진다")
print(f"  k=1 일 때 {context_chars[0]:.0f}자 → k=8 일 때 {context_chars[-1]:.0f}자")
print("  보통 3~5 에서 균형을 잡는다.")